# Logits Extractor Module

In [ ]:
def default_params(): 
    return {
        'current_model': 'M1',
        'gpu': True,
        'quantization': 'none', #['none',"int4", "int8", "float32", "float16"]
        'dataset': {
            'path': '/workspaces/CodeSmells/data/extension/mitigation/datasets',
            'current': 'base', # 'base' or 'prompted',
            'content_column': 'code',
            'sampling_size': 500,
            'prompt_column': 'prompt',
        },
        'logging_path': '/workspaces/CodeSmells/datax/code_smells/logs', 
        'callbacks_dir' : '/workspaces/CodeSmells/datax/code_smells/callbacks/mitigation',
        'cache_dir': '/workspaces/CodeSmells/datax/hugging_face_cache',
        'causal_models': {
            ##### BY ARCHITECTURE, SAME SIZE #####
            'M1' : 'codellama/CodeLlama-7b-hf', #https://huggingface.co/codellama/CodeLlama-7b-hf, 
            'M2' : 'mistralai/Mistral-7B-v0.3', #https://huggingface.co/mistralai/Mistral-7B-v0.3,
            'M3' : 'Qwen/Qwen2.5-Coder-7B', #https://huggingface.co/Qwen/Qwen2.5-Coder-7B,
            'M4' : 'bigcode/starcoder2-7b', #https://huggingface.co/bigcode/starcoder2-7b,
            ##### BY SIZE, SAME ARCHITECTURE #####
            'S1' : 'Qwen/Qwen2.5-Coder-0.5B', #https://huggingface.co/Qwen/Qwen2.5-Coder-0.5B,
            'S2' : 'Qwen/Qwen2.5-Coder-1.5B', #https://huggingface.co/Qwen/Qwen2.5-Coder-1.5B,
            'S3' : 'Qwen/Qwen2.5-Coder-3B', #https://huggingface.co/Qwen/Qwen2.5-Coder-3B,
            'S4' : 'Qwen/Qwen2.5-Coder-7B', #https://huggingface.co/Qwen/Qwen2.5-Coder-7B,
        }
    }
params = default_params()


#### Imports

In [2]:
import pandas as pd
import os
import time
import numpy as np
import torch
import gc
import seaborn as sns
from scipy import stats
from statistics import NormalDist
import matplotlib.pyplot as plt

In [3]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import load_dataset

In [4]:
def create_folder(path):
    if not os.path.exists(path):
        os.makedirs(path)

In [5]:
# Define log file path
log_file = f"{params['logging_path']}/{params['current_model']}"
create_folder(log_file)
log_file += '/log.txt'

# Create the log file if it doesn't exist
if not os.path.exists(log_file):
    with open(log_file, 'w'): 
        pass  # Create an empty log file

In [6]:
import logging
logging.basicConfig(filename=log_file, format='%(asctime)s : %(levelname)s : %(message)s', level=logging.INFO)

#### GPU

In [7]:
! nvidia-smi

Wed Jul  2 15:31:39 2025       
+-----------------------------------------------------------------------------+
| NVIDIA-SMI 470.103.01   Driver Version: 470.103.01   CUDA Version: 12.3     |
|-------------------------------+----------------------+----------------------+
| GPU  Name        Persistence-M| Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp  Perf  Pwr:Usage/Cap|         Memory-Usage | GPU-Util  Compute M. |
|                               |                      |               MIG M. |
|===============================+======================+======================|
|   0  NVIDIA A100-PCI...  Off  | 00000000:61:00.0 Off |                    0 |
| N/A   35C    P0    34W / 250W |      4MiB / 40536MiB |      0%      Default |
|                               |                      |             Disabled |
+-------------------------------+----------------------+----------------------+
                                                                               
+-------

In [8]:
torch.__version__

'2.1.2+cu121'

In [9]:
device = torch.device("cuda:0" if torch.cuda.is_available() and params['gpu'] else "cpu")
device

device(type='cuda', index=0)

In [10]:
torch.cuda.memory_allocated()

0

## Logits Extractor
>
> Extracting Tensor Logits from a given Neural Code Model
>

#### Model Loading

In [11]:
def instantiate_llm(model_name:str, cache_dir:str):
     '''Instantiate AutoModelForCausalLM'''
     tokenizer = AutoTokenizer.from_pretrained(model_name, cache_dir = cache_dir, use_fast=True)
     logging.info("Loaded AutoTokenizer - " + model_name)
     model = None
     if params['quantization'] == 'int4':
          model = AutoModelForCausalLM.from_pretrained(model_name, cache_dir = cache_dir, load_in_4bit=True)
     elif params['quantization'] == 'int8':
          model = AutoModelForCausalLM.from_pretrained(model_name, cache_dir = cache_dir, load_in_8bit=True)
     elif params['quantization'] == 'float32':
          model = AutoModelForCausalLM.from_pretrained(model_name, cache_dir = cache_dir, torch_dtype=torch.float32)
     elif params['quantization'] == 'float16':
          model = AutoModelForCausalLM.from_pretrained(model_name, cache_dir = cache_dir, torch_dtype=torch.float16)
     else: 
          model = AutoModelForCausalLM.from_pretrained(model_name, cache_dir = cache_dir)
     logging.info("Loaded AutoModelForCausalLM - " + model_name)

     return tokenizer, model

In [12]:
tokenizer, model = instantiate_llm(params['causal_models'][params['current_model']], params['cache_dir'])

2025-07-02 15:31:40.880708: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1751470300.899160  989532 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1751470300.904917  989532 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-07-02 15:31:40.925034: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

In [13]:
model.config

MistralConfig {
  "architectures": [
    "MistralForCausalLM"
  ],
  "attention_dropout": 0.0,
  "bos_token_id": 1,
  "eos_token_id": 2,
  "head_dim": null,
  "hidden_act": "silu",
  "hidden_size": 4096,
  "initializer_range": 0.02,
  "intermediate_size": 14336,
  "max_position_embeddings": 32768,
  "model_type": "mistral",
  "num_attention_heads": 32,
  "num_hidden_layers": 32,
  "num_key_value_heads": 8,
  "rms_norm_eps": 1e-05,
  "rope_theta": 1000000.0,
  "sliding_window": null,
  "tie_word_embeddings": false,
  "torch_dtype": "float32",
  "transformers_version": "4.52.4",
  "use_cache": true,
  "vocab_size": 32768
}

In [14]:
model.to(device) #WARNING, Verify the device before assigning to memory

MistralForCausalLM(
  (model): MistralModel(
    (embed_tokens): Embedding(32768, 4096)
    (layers): ModuleList(
      (0-31): 32 x MistralDecoderLayer(
        (self_attn): MistralAttention(
          (q_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear(in_features=4096, out_features=1024, bias=False)
          (v_proj): Linear(in_features=4096, out_features=1024, bias=False)
          (o_proj): Linear(in_features=4096, out_features=4096, bias=False)
        )
        (mlp): MistralMLP(
          (gate_proj): Linear(in_features=4096, out_features=14336, bias=False)
          (up_proj): Linear(in_features=4096, out_features=14336, bias=False)
          (down_proj): Linear(in_features=14336, out_features=4096, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): MistralRMSNorm((4096,), eps=1e-05)
        (post_attention_layernorm): MistralRMSNorm((4096,), eps=1e-05)
      )
    )
    (norm): MistralRMSNorm((4096,), eps=1e-0

#### Dataset

In [15]:
df_dataset = pd.read_json(f"{params['dataset']['path']}/{params['dataset']['current']}.json")

In [16]:
df_dataset

,id,commit_id,repo,path,file_name,fun_name,commit_message,code,url,language,...,s_line,s_column,s_end_line,s_end_column,s_code,category,input_lenght,prompt_id,prompt,original_code
0,2758,0e9baa6d2c7a752eb32e6da2359470f95a3efeec,PySyft,packages/syft/src/syft/oblv/model.py,model.py,get_uploaded_datasets,Adding method to get datasets list,You are an expert software engineer who writes...,https://github.com/OpenMined/PySyft.git,Python,...,9,12,9,100,"raise Exception(""User cannot connect to this d...",Warning,361,P1,You are an expert software engineer who writes...,def get_uploaded_datasets(self):\n if l...
1,115917,6eb408a9973fbc24c973d6524dc34cb9b1e0ee05,mindsdb,mindsdb/api/mongo/responders/delete.py,delete.py,_result,del model interface,You are an expert software engineer who writes...,https://github.com/mindsdb/mindsdb.git,Python,...,10,12,10,129,"raise Exception(""For db.predictors.delete oper...",Warning,293,P1,You are an expert software engineer who writes...,"def _result(self, query, request_env, mindsdb_..."
2,2758,0e9baa6d2c7a752eb32e6da2359470f95a3efeec,PySyft,packages/syft/src/syft/oblv/model.py,model.py,get_uploaded_datasets,Adding method to get datasets list,You are an expert software engineer who writes...,https://github.com/OpenMined/PySyft.git,Python,...,6,12,6,190,"raise Exception(""Either proxy not running or n...",Warning,361,P1,You are an expert software engineer who writes...,def get_uploaded_datasets(self):\n if l...
4,115111,d725c063e3ac3ab919fc8ed5f969fef8bc478209,mindsdb,mindsdb/api/mysql/mysql_proxy/datahub/datanode...,integration_datanode.py,select,fix,You are an expert software engineer who writes...,https://github.com/mindsdb/mindsdb.git,Python,...,4,12,4,49,raise Exception(result.error_message),Warning,148,P1,You are an expert software engineer who writes...,"def select(self, query):\n result = sel..."
5,189674,e040bcacd38378386749db18aeba575b93f4ebca,manim,manim/mobject/geometry/arc.py,arc.py,get_tip,Improved structure of the :mod:`.mobject` modu...,You are an expert software engineer who writes...,https://github.com/ManimCommunity/manim.git,Python,...,4,12,4,44,"raise Exception(""tip not found"")",Warning,54,P1,You are an expert software engineer who writes...,def get_tip(self):\n \n tips = s...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
494,113833,5222e094c7c3353d17d509a259efdf2f53cc2fac,mindsdb,mindsdb/api/mysql/mysql_proxy/classes/sql_stat...,sql_statement_parser.py,get_keyword,'create table' alias for 'create predictor',You are an expert software engineer who writes...,https://github.com/mindsdb/mindsdb.git,Python,...,29,12,29,62,raise Exception('Cant get keyword from stateme...,Warning,425,P1,You are an expert software engineer who writes...,def get_keyword(sql):\n \n START...
495,115229,4f2861b6ded274d4a41322c107ace8107e86ebea,mindsdb,mindsdb/interfaces/database/views.py,views.py,add,store integration in sql of view (before save it),You are an expert software engineer who writes...,https://github.com/mindsdb/mindsdb.git,Python,...,21,16,21,88,"raise Exception(f""Can't find integration with ...",Warning,264,P1,You are an expert software engineer who writes...,"def add(self, name, query, integration_name, c..."
497,2842,ccd4b9330a186090cc87e94d2da1093d45de329f,PySyft,packages/syft/src/syft/oblv/model.py,model.py,request_publish,Changes for model,You are an expert software engineer who writes...,https://github.com/OpenMined/PySyft.git,Python,...,2,12,2,116,"raise Exception(""No Domain Clients added. Set ...",Warning,497,P1,You are an expert software engineer who writes...,"def request_publish(self, dataset_id, sigma = ..."
498,116308,326622a6fb33664de21ee1627f5f083f11b59a9e,mindsdb,mindsdb/integrations/handlers/ludwig_handler/l...,ludwig_handler.py,_learn,fix: add hyperopt,You are an expert software engineer who writes...,https://github.com/mindsdb/mindsdb.git,Python,...,8,12,8,85,"raise Exception(""Ludwig handler does not suppo...",Warning,466,P1,You

In [17]:
#df_dataset = df_dataset[df_dataset['input_lenght']>=700]
#df_dataset = df_dataset[:20]

#### Logit Inference

In [18]:
def logit_extractor(model, batch, tf_encoded_inputs, from_index=0):
    """
    Output is the class CausalLMOutputWithPast (https://huggingface.co/transformers/v4.10.1/main_classes/output.html?highlight=causallmoutputwithpast)"
    logits (torch.FloatTensor of shape (batch_size, sequence_length, config.vocab_size)) – Prediction scores of the language modeling head (scores for each vocabulary token before SoftMax).
    The expression i.type(torch.LongTensor).to(device) is for casting labels for the loss
    """
    callbacks_dir = f"{params['callbacks_dir']}/{params['dataset']['current']}/{params['current_model']}_q_{params['quantization']}"
    create_folder(callbacks_dir)
    
    for idx, n in enumerate(range(from_index, len(tf_encoded_inputs), batch)):
        torch.cuda.empty_cache()
        output = []
        for encoded_sample in tf_encoded_inputs[n:n+batch]:
            output.append( 
                model(input_ids = encoded_sample, labels = encoded_sample)
            )
        output_logits = [ o['logits'].detach().to('cpu').numpy() for o in output ]  #Logits Extraction
        output_loss = np.array([ o.loss.detach().to('cpu').numpy() for o in output ])  #Language modeling loss (for next-token prediction).

        #Saving Callbacks
        current_batch = idx + (from_index//batch)
        for jdx, o_logits in enumerate(output_logits):
            np.save(f"{callbacks_dir}/logits_tensor[{jdx+n}]_batch[{current_batch}].npy", o_logits)
        np.save(f"{callbacks_dir}/_loss_batch[{current_batch}].npy", output_loss)
        
        print(f"Batch [{current_batch}] Completed")

        #Memory Released
        for out in output:
            del out.logits
            torch.cuda.empty_cache()
            del out.loss
            torch.cuda.empty_cache()
        for out in output_logits:
            del out
            torch.cuda.empty_cache()
        for out in output_loss:
            del out
            torch.cuda.empty_cache()

In [19]:
#Casting Integers to Tensor Integers. Make sure the tesor is created in a device
#We ignored the parameter attention_mask since we are not using masking here [https://huggingface.co/transformers/v4.10.1/glossary.html#attention-mask]
tf_encoded_inputs = [tokenizer(sample, return_tensors='pt')['input_ids'].to(device) for sample in df_dataset[params['dataset']['content_column']].values]

In [20]:
## ACTUAL EXPERIMENT
## TIME AND MEMORY CONSUMING
logit_extractor(
    model = model,
    batch = 1, 
    tf_encoded_inputs = tf_encoded_inputs, 
    from_index=0
)

Batch [0] Completed
Batch [1] Completed
Batch [2] Completed
Batch [3] Completed
Batch [4] Completed
Batch [5] Completed
Batch [6] Completed
Batch [7] Completed
Batch [8] Completed
Batch [9] Completed
Batch [10] Completed
Batch [11] Completed
Batch [12] Completed
Batch [13] Completed
Batch [14] Completed
Batch [15] Completed
Batch [16] Completed
Batch [17] Completed
Batch [18] Completed
Batch [19] Completed
Batch [20] Completed
Batch [21] Completed
Batch [22] Completed
Batch [23] Completed
Batch [24] Completed
Batch [25] Completed
Batch [26] Completed
Batch [27] Completed
Batch [28] Completed
Batch [29] Completed
Batch [30] Completed
Batch [31] Completed
Batch [32] Completed
Batch [33] Completed
Batch [34] Completed
Batch [35] Completed
Batch [36] Completed
Batch [37] Completed
Batch [38] Completed
Batch [39] Completed
Batch [40] Completed
Batch [41] Completed
Batch [42] Completed
Batch [43] Completed
Batch [44] Completed
Batch [45] Completed
Batch [46] Completed
Batch [47] Completed
Ba

In [21]:
print("================================= PROCESS COMPLETE =================================")

================================= PROCESS COMPLETE =================================


In [22]:
del model
torch.cuda.empty_cache()
gc.collect()

20